In [1]:
import re
import pandas as pd

INPUT_CSV = "./version1.csv"
OUTPUT_CSV = "Nazario_5_preprocessed.csv"

HEADER_PATTERN = re.compile(r'^(from|sent|to|cc|bcc|subject|date):\s*', re.I)
REPLY_MARKER_PATTERN = re.compile(r'^(-----Original Message-----|_{3,}|-{3,}|on .+wrote:)$', re.I)
SIGNOFF_PATTERN = re.compile(r'^(best|regards|thanks|thank you|sincerely|cheers)[,!]?\s*$', re.I)


def normalize_entities(text: str) -> str:
    text = re.sub(r'https?://\S+|www\.\S+', ' <URL> ', text)
    text = re.sub(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b', ' <EMAIL> ', text)
    text = re.sub(r'(?:(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{3,4})', ' <PHONE> ', text)
    text = re.sub(
        r'(?i)\b(user\s*id|userid|username|login|pw|pwd|password)\s*[:=]\s*\S+',
        lambda m: m.group(1).upper().replace(' ', '_') + ': <TOKEN>',
        text,
    )
    return text


def clean_subject(text: str) -> str:
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'^(\s*(re|fw|fwd)\s*:\s*)+', '', text, flags=re.I)
    text = normalize_entities(text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text


def clean_body(text: str) -> str:
    if pd.isna(text):
        return ''

    text = str(text).replace('\r\n', '\n').replace('\r', '\n')
    lines = text.split('\n')
    cleaned_lines = []

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if REPLY_MARKER_PATTERN.match(line):
            break

        if HEADER_PATTERN.match(line):
            i += 1
            continue

        if SIGNOFF_PATTERN.match(line):
            break

        if re.fullmatch(r'[_\-]{3,}', line):
            i += 1
            continue

        cleaned_lines.append(lines[i])
        i += 1

    cleaned_text = '\n'.join(cleaned_lines)
    cleaned_text = normalize_entities(cleaned_text)
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
    cleaned_text = re.sub(r'[ \t]{2,}', ' ', cleaned_text).strip()
    return cleaned_text


def preprocess_email_csv(input_csv: str, output_csv: str):
    df = pd.read_csv(input_csv)

    if 'subject' not in df.columns or 'body' not in df.columns:
        raise ValueError("CSV must contain 'subject' and 'body' columns")

    df['clean_subject'] = df['subject'].apply(clean_subject)
    df['clean_body'] = df['body'].apply(clean_body)

    df.to_csv(output_csv, index=False)
    print(f"Saved cleaned file to: {output_csv}")


if __name__ == '__main__':
    preprocess_email_csv(INPUT_CSV, OUTPUT_CSV)

Saved cleaned file to: Nazario_5_preprocessed.csv


Cleaned 2


In [2]:
import re
import unicodedata
import pandas as pd

INPUT_CSV = "./Nazario_5_preprocessed.csv"
OUTPUT_CSV = "Nazario_5_preprocessed_v2.csv"

SPACE_RE = re.compile(r'[ \t]+')
MULTI_NL_RE = re.compile(r'\n{3,}')
URL_RE = re.compile(r'https?://\S+|www\.\S+', re.I)
EMAIL_RE = re.compile(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b')
PHONE_RE = re.compile(r'(?:(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{3,4})')
HEADER_RE = re.compile(r'^(from|sent|to|cc|bcc|subject|date):\s*', re.I)
REPLY_RE = re.compile(r'^(-----Original Message-----|_{3,}|-{3,}|on .+wrote:)$', re.I)
SIGNOFF_RE = re.compile(r'^(best|regards|thanks|thank you|sincerely|cheers)[,!]?\s*$', re.I)
TOKEN_RE = re.compile(r'(?i)\b(user\s*id|userid|username|login|pw|pwd|password)\s*[:=]\s*\S+')

def normalize_unicode(text: str) -> str:
    return unicodedata.normalize('NFKC', text)

def normalize_entities(text: str) -> str:
    text = URL_RE.sub(' <URL> ', text)
    text = EMAIL_RE.sub(' <EMAIL> ', text)
    text = PHONE_RE.sub(' <PHONE> ', text)
    text = TOKEN_RE.sub(lambda m: m.group(1).upper().replace(' ', '_') + ': <TOKEN>', text)
    return text

def normalize_subject(text: str) -> str:
    if pd.isna(text):
        return ''
    text = normalize_unicode(str(text))
    text = re.sub(r'^(\s*(re|fw|fwd)\s*:\s*)+', '', text, flags=re.I)
    text = normalize_entities(text)
    text = SPACE_RE.sub(' ', text).strip()
    return text

def normalize_body(text: str) -> str:
    if pd.isna(text):
        return ''
    text = normalize_unicode(str(text)).replace('\r\n', '\n').replace('\r', '\n')
    lines = []
    for raw in text.split('\n'):
        line = raw.strip()
        if not line:
            lines.append('')
            continue
        if REPLY_RE.match(line):
            break
        if HEADER_RE.match(line):
            continue
        if SIGNOFF_RE.match(line):
            break
        if re.fullmatch(r'[_\-]{3,}', line):
            continue
        line = normalize_entities(line)
        line = SPACE_RE.sub(' ', line).strip()
        if line:
            lines.append(line)
    text = '\n'.join(lines)
    text = MULTI_NL_RE.sub('\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text).strip()
    return text

df = pd.read_csv(INPUT_CSV)
df['subject_cleaned_2'] = df['subject'].apply(normalize_subject)
df['body_cleaned_2'] = df['body'].apply(normalize_body)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")

Saved: Nazario_5_preprocessed_v2.csv
